In [3]:
from google.colab import files

print('Upload your Hindi audio file (.wav or .mp3)')
uploaded = files.upload()
audio_filename = list(uploaded.keys())[0]
print(f'✅ Uploaded: {audio_filename}')

# GROUND TRUTH: type what you actually said (for rough WER calculation)
# Change this to match your recording!
GROUND_TRUTH = 'वाराणसी में सबसे प्रसिद्ध घाट कौन सा है'

Upload your Hindi audio file (.wav or .mp3)


Saving Audio10.mp3 to Audio10.mp3
✅ Uploaded: Audio10.mp3


In [4]:
# !pip install -q openai-whisper sentence-transformers faiss-cpu gtts
!pip install -q faster-whisper sentence-transformers faiss-cpu gtts
print('✅ All packages installed: whisper, sentence-transformers, faiss-cpu, gtts')

✅ All packages installed: whisper, sentence-transformers, faiss-cpu, gtts


In [5]:
from faster_whisper import WhisperModel
import time

print('Loading faster-whisper medium (INT8)...')
model = WhisperModel("medium", compute_type="int8")

t0 = time.time()
segments, info = model.transcribe(audio_filename, language="hi", task="transcribe")
transcribed_query = " ".join([seg.text for seg in segments]).strip()
t_faster = time.time() - t0

print(f'Transcription ({t_faster:.2f}s): {transcribed_query}')

Loading faster-whisper medium (INT8)...
Transcription (1.20s): वारनसी में सबसे प्रसिद घात कौनसा है?


In [7]:
def word_error_rate(reference, hypothesis):
    ref_words = reference.strip().split()
    hyp_words = hypothesis.strip().split()

    d = [[0] * (len(hyp_words) + 1) for _ in range(len(ref_words) + 1)]
    for i in range(len(ref_words) + 1):
        d[i][0] = i
    for j in range(len(hyp_words) + 1):
        d[0][j] = j
    for i in range(1, len(ref_words) + 1):
        for j in range(1, len(hyp_words) + 1):
            cost = 0 if ref_words[i-1] == hyp_words[j-1] else 1
            d[i][j] = min(d[i-1][j] + 1,
                          d[i][j-1] + 1,
                          d[i-1][j-1] + cost)

    return round(d[len(ref_words)][len(hyp_words)] / len(ref_words) * 100, 1)

# faster-whisper only has one output now (transcribed_query)
wer_faster = word_error_rate(GROUND_TRUTH, transcribed_query)

print('\n' + '='*50)
print('📏 WORD ERROR RATE')
print('='*50)
print(f'Ground truth      : {GROUND_TRUTH}')
print(f'faster-whisper    : {transcribed_query}')
print(f'faster-whisper WER: {wer_faster}%')
print('='*50)


📏 WORD ERROR RATE
Ground truth      : वाराणसी में सबसे प्रसिद्ध घाट कौन सा है
faster-whisper    : वारनसी में सबसे प्रसिद घात कौनसा है?
faster-whisper WER: 75.0%


In [38]:

contexts = [
    # Varanasi / Ghats
    'वाराणसी का सबसे प्रसिद्ध और मुख्य घाट दशाश्वमेध घाट है।',
    'वाराणसी में सबसे प्रसिद्ध घाट का नाम दशाश्वमेध घाट है।',
    'वाराणसी में सबसे प्रसिद्ध घाट दशाश्वमेध घाट है, जहाँ प्रतिदिन गंगा आरती होती है।',
    'गंगा आरती वाराणसी में हर शाम दशाश्वमेध घाट पर होती है।',
    'मणिकर्णिका घाट वाराणसी का सबसे पवित्र श्मशान घाट है।',
    'वाराणसी में 84 घाट गंगा नदी के किनारे स्थित हैं।',
    'वाराणसी को काशी और बनारस के नाम से भी जाना जाता है — यह भारत का सबसे पुराना शहर है।',
    'काशी विश्वनाथ मंदिर वाराणसी में स्थित है और भगवान शिव को समर्पित है।',
    # Ramayana / Hanuman
    'हनुमान जी भगवान राम के परम भक्त और सेवक थे।',
    'हनुमान जी ने लंका में जाकर माता सीता की खोज की और उन्हें राम जी का संदेश दिया।',
    'हनुमान जी ने संजीवनी बूटी लाकर लक्ष्मण जी की जान बचाई थी।',
    'रावण ने माता सीता का अपहरण किया और उन्हें लंका ले गया।',
    'भगवान राम ने वानर सेना की मदद से रावण का वध किया।',
    # Ganga / Nature
    'गंगा नदी हिमालय के गंगोत्री ग्लेशियर से निकलती है।',
    'गंगा नदी को भारत की सबसे पवित्र नदी माना जाता है।',
    # Weather / General
    'वाराणसी में गर्मियों में तापमान 45 डिग्री तक पहुँच सकता है।',
    'मानसून जून से सितंबर तक उत्तर प्रदेश में रहता है।',
    # BHU
    'वाराणसी में बनारस हिंदू विश्वविद्यालय यानी BHU स्थित है।',
'BHU यानी बनारस हिंदू विश्वविद्यालय वाराणसी का सबसे प्रसिद्ध विश्वविद्यालय है।',
'वाराणसी में स्थित बनारस हिंदू विश्वविद्यालय एशिया के सबसे बड़े विश्वविद्यालयों में से एक है।',

    'बनारस हिंदू विश्वविद्यालय (BHU) वाराणसी में स्थित है और एशिया के सबसे बड़े विश्वविद्यालयों में से एक है।',

'अयोध्या भगवान राम की जन्मभूमि है और उत्तर प्रदेश में स्थित है।',
'प्रयागराज में गंगा, यमुना और सरस्वती नदियों का संगम होता है।',
'काशी में मरने वाले को मोक्ष मिलता है — यह हिंदू मान्यता है।',
'वाराणसी का रेशमी कपड़ा और बनारसी साड़ी विश्व प्रसिद्ध है।',
'रामचरितमानस की रचना तुलसीदास ने वाराणसी में की थी।',
'सारनाथ वाराणसी के पास स्थित है जहाँ बुद्ध ने पहला उपदेश दिया था।',
'इलाहाबाद विश्वविद्यालय भारत के सबसे पुराने विश्वविद्यालयों में से एक है।',
'उत्तर प्रदेश भारत का सबसे अधिक जनसंख्या वाला राज्य है।',
'गंगा आरती वाराणसी में हर शाम दशाश्वमेध घाट पर होती है।',
]

print(f'✅ Knowledge base: {len(contexts)} sentences loaded')

✅ Knowledge base: 30 sentences loaded


In [39]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print('Loading sentence embedder (all-MiniLM-L6-v2)...')
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print('Encoding knowledge base...')
t0 = time.time()
embeddings = embedder.encode(contexts, normalize_embeddings=True, show_progress_bar=True)
embeddings = np.array(embeddings, dtype='float32')
t_index = time.time() - t0

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f'✅ FAISS index built in {t_index:.2f}s | {index.ntotal} vectors | dim={dim}')

Loading sentence embedder (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding knowledge base...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS index built in 0.05s | 30 vectors | dim=384


In [40]:
def retrieve(query, k=2):
    """Retrieve top-k most relevant sentences for a Hindi query."""
    t0 = time.time()
    q_emb = embedder.encode([query], show_progress_bar=False)
    q_emb = np.array(q_emb, dtype='float32')
    distances, indices = index.search(q_emb, k=k)
    latency_ms = (time.time() - t0) * 1000

    results = [(contexts[i], distances[0][rank]) for rank, i in enumerate(indices[0])]
    return results, latency_ms

# Quick sanity check
test_results, lat = retrieve('हनुमान जी ने क्या किया?', k=2)
print(f'Sanity check retrieval ({lat:.1f}ms):')
for ctx, dist in test_results:
    print(f'  dist={dist:.3f} | {ctx}')

Sanity check retrieval (14.3ms):
  dist=10.971 | हनुमान जी भगवान राम के परम भक्त और सेवक थे।
  dist=11.124 | सारनाथ वाराणसी के पास स्थित है जहाँ बुद्ध ने पहला उपदेश दिया था।


In [41]:
from gtts import gTTS
from IPython.display import Audio, display

def speak_response(response_text, filename='response.mp3'):
    """Convert Hindi text to speech and play it in Colab."""
    tts = gTTS(text=response_text, lang='hi', slow=False)
    tts.save(filename)
    print(f'🔊 Playing: "{response_text}"')
    display(Audio(filename, autoplay=True))
    return filename

# Test it
speak_response('नमस्ते! आपका प्रश्न सुना गया है। उत्तर खोजा जा रहा है।', 'greeting.mp3')

🔊 Playing: "नमस्ते! आपका प्रश्न सुना गया है। उत्तर खोजा जा रहा है।"


'greeting.mp3'

In [42]:
print('=' * 65)
print('🎙️  HINDI VOICE-TO-VOICE RAG DEMO')
print('=' * 65)

# STAGE 1: ASR
print(f'\n[1/3] 🎙️  INPUT AUDIO : {audio_filename}')
print(f'      📝 TRANSCRIBED : {transcribed_query}')

# STAGE 2: Retrieval
top_results, retrieval_ms = retrieve(transcribed_query, k=2)
best_context, best_dist = top_results[0]

print(f'\n[2/3] 🔍 RETRIEVAL   ({retrieval_ms:.1f}ms)')
for rank, (ctx, dist) in enumerate(top_results, 1):
    print(f'      [{rank}] dist={dist:.3f} | {ctx}')

# STAGE 3: TTS Response
# Build a natural-sounding Hindi response
response_text = f'{best_context}'

print(f'\n[3/3] 🔊 SPOKEN RESPONSE:')
print(f'      "{response_text}"')
print()

speak_response(response_text, 'final_response.mp3')

print('\n' + '='*65)
print('✅ Pipeline complete')
print(f'   ASR (faster-whisper INT8) : {t_faster:.2f}s')
print(f'   Retrieval (FAISS)         : {retrieval_ms:.1f}ms')
print(f'   TTS (gTTS)                : ~1–2s (network call)')
print('='*65)

🎙️  HINDI VOICE-TO-VOICE RAG DEMO

[1/3] 🎙️  INPUT AUDIO : Audio10.mp3
      📝 TRANSCRIBED : वारनसी में सबसे प्रसिद घात कौनसा है?

[2/3] 🔍 RETRIEVAL   (15.4ms)
      [1] dist=9.675 | वाराणसी का सबसे प्रसिद्ध और मुख्य घाट दशाश्वमेध घाट है।
      [2] dist=9.905 | वाराणसी में सबसे प्रसिद्ध घाट का नाम दशाश्वमेध घाट है।

[3/3] 🔊 SPOKEN RESPONSE:
      "वाराणसी का सबसे प्रसिद्ध और मुख्य घाट दशाश्वमेध घाट है।"

🔊 Playing: "वाराणसी का सबसे प्रसिद्ध और मुख्य घाट दशाश्वमेध घाट है।"



✅ Pipeline complete
   ASR (faster-whisper INT8) : 1.20s
   Retrieval (FAISS)         : 15.4ms
   TTS (gTTS)                : ~1–2s (network call)


In [43]:
test_queries = [
    'वाराणसी में सबसे प्रसिद्ध घाट कौन सा है?',
    'हनुमान जी ने राम जी की मदद कैसे की?',
    'गंगा नदी कहाँ से निकलती है?',
    'बनारस में कौन सा विश्वविद्यालय है?',
]

print('='*65)
print('🧪 BATCH RETRIEVAL TEST')
print('='*65)
for q in test_queries:
    results, lat = retrieve(q, k=1)
    print(f'Q: {q}')
    print(f'A: {results[0][0]}  ({lat:.1f}ms)')
    print()

🧪 BATCH RETRIEVAL TEST
Q: वाराणसी में सबसे प्रसिद्ध घाट कौन सा है?
A: वाराणसी का सबसे प्रसिद्ध और मुख्य घाट दशाश्वमेध घाट है।  (20.5ms)

Q: हनुमान जी ने राम जी की मदद कैसे की?
A: हनुमान जी भगवान राम के परम भक्त और सेवक थे।  (17.2ms)

Q: गंगा नदी कहाँ से निकलती है?
A: गंगा नदी हिमालय के गंगोत्री ग्लेशियर से निकलती है।  (14.1ms)

Q: बनारस में कौन सा विश्वविद्यालय है?
A: वाराणसी में स्थित बनारस हिंदू विश्वविद्यालय एशिया के सबसे बड़े विश्वविद्यालयों में से एक है।  (15.0ms)



In [44]:
import torch
import time
import numpy as np

# Measure avg retrieval latency
N = 100
q_emb = embedder.encode(['हनुमान जी कौन थे?'], show_progress_bar=False)
q_emb = np.array(q_emb, dtype='float32')
t0 = time.time()
for _ in range(N):
    index.search(q_emb, k=1)
avg_retrieval_ms = (time.time() - t0) / N * 1000

print('\n' + '='*65)
print('📊 PERFORMANCE SUMMARY')
print('='*65)
print(f'Hardware                    : {"GPU (T4)" if torch.cuda.is_available() else "CPU"}')
print(f'faster-whisper medium INT8  : 1.20s  (was 2.2s with openai-whisper)')
print(f'Speed improvement           : ~45% faster')
print(f'faster-whisper WER          : {wer_faster}%')
print(f'FAISS retrieval (avg {N}x)   : {avg_retrieval_ms:.2f}ms')
print(f'Knowledge base size         : {len(contexts)} sentences')
print(f'Correct answer retrieved    : दशाश्वमेध घाट ✅')
print('='*65)


📊 PERFORMANCE SUMMARY
Hardware                    : GPU (T4)
faster-whisper medium INT8  : 1.20s  (was 2.2s with openai-whisper)
Speed improvement           : ~45% faster
faster-whisper WER          : 75.0%
FAISS retrieval (avg 100x)   : 0.01ms
Knowledge base size         : 30 sentences
Correct answer retrieved    : दशाश्वमेध घाट ✅


In [45]:
!pip install -q gradio

import gradio as gr
from gtts import gTTS

def hindi_pipeline(audio_path):
    if audio_path is None:
        return "No audio received", None

    # ASR
    import time
    t0 = time.time()
    segments, _ = model.transcribe(audio_path, language="hi", task="transcribe")
    query = " ".join([s.text for s in segments]).strip()
    asr_time = time.time() - t0

    # Retrieval
    t0 = time.time()
    q_emb = np.array(embedder.encode([query], normalize_embeddings=True), dtype='float32')
    distances, indices = index.search(q_emb, k=2)
    retrieval_ms = (time.time() - t0) * 1000
    top_context = contexts[indices[0][0]]

    # TTS
    tts = gTTS(text=top_context, lang='hi')
    tts.save("gradio_response.mp3")

    summary = f"""🎙️ Query: {query}
📚 Retrieved: {top_context}
⏱️ ASR: {asr_time:.2f}s | Retrieval: {retrieval_ms:.1f}ms"""

    return summary, "gradio_response.mp3"

demo = gr.Interface(
    fn=hindi_pipeline,
    inputs=gr.Audio(type="filepath", label="🎙️ Speak in Hindi"),
    outputs=[
        gr.Textbox(label="Pipeline Output"),
        gr.Audio(label="🔊 Hindi Response")
    ],
    title="🎙️ Hindi Voice-to-Voice RAG",
    description="Speak a Hindi question → Whisper ASR → FAISS retrieval → gTTS spoken response"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d4b98ecbf2b8ba517f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [50]:
!pip install -q huggingface_hub

from huggingface_hub import notebook_login
notebook_login()

In [51]:
app_code = '''
import gradio as gr
from faster_whisper import WhisperModel
from sentence_transformers import SentenceTransformer
import faiss, numpy as np
from gtts import gTTS
import time

# Load models
model = WhisperModel("medium", compute_type="int8")
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

contexts = [
    "वाराणसी का सबसे प्रसिद्ध और मुख्य घाट दशाश्वमेध घाट है।",
    "वाराणसी में सबसे प्रसिद्ध घाट दशाश्वमेध घाट है, जहाँ प्रतिदिन गंगा आरती होती है।",
    "गंगा आरती वाराणसी में हर शाम दशाश्वमेध घाट पर होती है।",
    "वाराणसी में बनारस हिंदू विश्वविद्यालय यानी BHU स्थित है।",
    "BHU यानी बनारस हिंदू विश्वविद्यालय वाराणसी का सबसे प्रसिद्ध विश्वविद्यालय है।",
    "मणिकर्णिका घाट वाराणसी का सबसे पवित्र श्मशान घाट है।",
    "वाराणसी में 84 घाट गंगा नदी के किनारे स्थित हैं।",
    "वाराणसी को काशी और बनारस के नाम से भी जाना जाता है।",
    "काशी विश्वनाथ मंदिर वाराणसी में स्थित है और भगवान शिव को समर्पित है।",
    "हनुमान जी भगवान राम के परम भक्त और सेवक थे।",
    "हनुमान जी ने लंका में जाकर माता सीता की खोज की।",
    "हनुमान जी ने संजीवनी बूटी लाकर लक्ष्मण जी की जान बचाई थी।",
    "रावण ने माता सीता का अपहरण किया और उन्हें लंका ले गया।",
    "भगवान राम ने वानर सेना की मदद से रावण का वध किया।",
    "गंगा नदी हिमालय के गंगोत्री ग्लेशियर से निकलती है।",
    "सारनाथ वाराणसी के पास स्थित है जहाँ बुद्ध ने पहला उपदेश दिया था।",
    "रामचरितमानस की रचना तुलसीदास ने वाराणसी में की थी।",
    "प्रयागराज में गंगा, यमुना और सरस्वती नदियों का संगम होता है।",
    "अयोध्या भगवान राम की जन्मभूमि है और उत्तर प्रदेश में स्थित है।",
    "वाराणसी का रेशमी कपड़ा और बनारसी साड़ी विश्व प्रसिद्ध है।",
]

embeddings = np.array(embedder.encode(contexts, normalize_embeddings=True), dtype="float32")
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

def hindi_pipeline(audio_path):
    if audio_path is None:
        return "No audio received", None
    t0 = time.time()
    segments, _ = model.transcribe(audio_path, language="hi", task="transcribe")
    query = " ".join([s.text for s in segments]).strip()
    asr_time = time.time() - t0

    t0 = time.time()
    q_emb = np.array(embedder.encode([query], normalize_embeddings=True), dtype="float32")
    distances, indices = index.search(q_emb, k=2)
    retrieval_ms = (time.time() - t0) * 1000
    top_context = contexts[indices[0][0]]

    tts = gTTS(text=top_context, lang="hi")
    tts.save("response.mp3")

    summary = f"""🎙️ Query: {query}
📚 Retrieved: {top_context}
⏱️ ASR: {asr_time:.2f}s | Retrieval: {retrieval_ms:.1f}ms"""
    return summary, "response.mp3"

demo = gr.Interface(
    fn=hindi_pipeline,
    inputs=gr.Audio(type="filepath", label="🎙️ Speak in Hindi"),
    outputs=[
        gr.Textbox(label="Pipeline Output"),
        gr.Audio(label="🔊 Hindi Response")
    ],
    title="🎙️ Hindi Voice-to-Voice RAG",
    description="Speak Hindi → Whisper ASR → FAISS retrieval → gTTS spoken response | Built by Vedant Shri Agarwal"
)

demo.launch()
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py created")

✅ app.py created


In [52]:
req = """faster-whisper
sentence-transformers
faiss-cpu
gtts
gradio"""

with open("requirements.txt", "w") as f:
    f.write(req)

print("✅ requirements.txt created")

✅ requirements.txt created


In [53]:
from huggingface_hub import HfApi
api = HfApi()

api.create_repo(
    repo_id="hindi-voice-rag",
    repo_type="space",
    space_sdk="gradio",
    exist_ok=True
)

api.upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id=f"{api.whoami()['name']}/hindi-voice-rag",
    repo_type="space"
)

api.upload_file(
    path_or_fileobj="requirements.txt",
    path_in_repo="requirements.txt",
    repo_id=f"{api.whoami()['name']}/hindi-voice-rag",
    repo_type="space"
)

print("✅ Deployed!")
print(f"URL: https://huggingface.co/spaces/{api.whoami()['name']}/hindi-voice-rag")

✅ Deployed!
URL: https://huggingface.co/spaces/Vedant0527/hindi-voice-rag
